In [4]:
!pip install pandas scikit-learn

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score

In [5]:
from datasets import load_dataset
import pandas as pd

#load dataset
ds = load_dataset("google/civil_comments")

#convert to pandas
train_df = ds["train"].to_pandas()
test_df = ds["test"].to_pandas()

#create binary labels (toxicity > 0.5)
train_df["label"] = (train_df["toxicity"] > 0.5).astype(int)
test_df["label"] = (test_df["toxicity"] > 0.5).astype(int)

#keep only relevant columns for now
train_df = train_df[["text", "label"]]
test_df = test_df[["text", "label"]]

#drop missing values
train_df = train_df.dropna()
test_df = test_df.dropna()

#preview
train_df.head()

#basic stats
print("Train shape:", train_df.shape)
print(train_df["label"].value_counts())

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00002.parquet:   0%|          | 0.00/194M [00:00<?, ?B/s]

data/train-00001-of-00002.parquet:   0%|          | 0.00/187M [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/20.8M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1804874 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/97320 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/97320 [00:00<?, ? examples/s]

Train shape: (1804874, 2)
label
0    1698436
1     106438
Name: count, dtype: int64


In [7]:
#define features
X_train = train_df["text"]
y_train = train_df["label"]

X_test = test_df["text"]
y_test = test_df["label"]

In [8]:
vectorizer = TfidfVectorizer(
    max_features=10000,
    stop_words='english'
)

X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

In [9]:
#train model
model = LogisticRegression(max_iter=1000)
model.fit(X_train_vec, y_train)


#predict
y_pred = model.predict(X_test_vec)


#evaluate
print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred))

#save results
results = pd.DataFrame({
    "text": X_test,
    "true": y_test,
    "pred": y_pred
})

# Errors
errors = results[results['true'] != results['pred']]
errors.head(10)

Accuracy: 0.9593094944512947

Classification Report:

              precision    recall  f1-score   support

           0       0.97      0.99      0.98     91532
           1       0.78      0.44      0.56      5788

    accuracy                           0.96     97320
   macro avg       0.87      0.72      0.77     97320
weighted avg       0.95      0.96      0.95     97320



,text,true,pred
26,"Ignorance is bliss, ain't it?",0,1
36,The ignorance and bigotry comes from your post!,1,0
49,"An ""abject lesson"" is a lesson that is painful...",1,0
51,Right on the money Gary Crum. And if they hide...,1,0
84,Somebody needs to dig up the Peterson's back y...,1,0
86,Only in America can you get 25 likes for admit...,1,0
112,"Why? And facts, actual policies and action rat...",0,1
135,"""Just saw headlines on a couple of sites where...",1,0
136,"Who cares? In all seriousness she, as with al...",1,0
197,You know? I've been listening to your sillines...,1,0
